# **7. Визуализация и анализ результатов моделей (Visualization and Model Result Analysis)**

* __Цель:__ визуально исследовать результаты моделей на validation-выборке, используя воспроизводимые выходные данные экспериментального pipeline.
* __Задачи:__
  * загрузить сохранённые validation predictions, metrics и comparison table;
  * визуализировать временной ряд, прогнозы и ошибки selected-модели;
  * сопоставить модели по validation RMSE;
  * извлечь transformed feature names и importance из сохранённого CatBoost pipeline;
  * подтвердить отсутствие test outputs в визуальном сравнении.
* __Алгоритм выполнения:__
  1. Проверить наличие выходных данных экспериментального pipeline.
  2. Выбрать validation predictions модели `catboost`.
  3. Построить временные и диагностические графики прогноза.
  4. Визуализировать почасовую ошибку и validation-сравнение моделей.
  5. Построить feature importance на основе сохранённого fitted pipeline.
  6. Выполнить методологический аудит и сформулировать краткую интерпретацию.

In [ ]:
import joblib
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from traffic_forecasting.config import (
    ACTUAL_VS_PREDICTED_FIGURE_PATH,
    ERROR_BY_HOUR_FIGURE_PATH,
    EXPERIMENT_COMPARISON_PATH,
    EXPERIMENT_METRICS_PATH,
    FEATURE_IMPORTANCE_FIGURE_PATH,
    MODEL_COMPARISON_FIGURE_PATH,
    MODELS_DIR,
    RESIDUAL_DISTRIBUTION_FIGURE_PATH,
    TRAFFIC_VOLUME_TIME_SERIES_FIGURE_PATH,
    TRUE_VS_PREDICTED_FIGURE_PATH,
    VALIDATION_PREDICTIONS_PATH,
)
from traffic_forecasting.visualization import (
    add_prediction_errors,
    extract_feature_importance,
    plot_actual_vs_predicted,
    plot_error_by_hour,
    plot_feature_importance,
    plot_model_comparison,
    plot_residual_distribution,
    plot_traffic_volume_time_series,
    plot_true_vs_predicted,
    summarize_error_by_hour,
)

SELECTED_MODEL = "catboost"
SELECTED_PIPELINE_PATH = MODELS_DIR / "catboost_pipeline.joblib"
STAGE_9_INPUTS = (
    EXPERIMENT_METRICS_PATH,
    EXPERIMENT_COMPARISON_PATH,
    VALIDATION_PREDICTIONS_PATH,
    SELECTED_PIPELINE_PATH,
)

missing_inputs = [path for path in STAGE_9_INPUTS if not path.is_file()]
if missing_inputs:
    missing_text = ", ".join(str(path) for path in missing_inputs)
    raise FileNotFoundError(
        "Missing experiment pipeline outputs: "
        f"{missing_text}. Run scripts/run_experiments.py first."
    )

## **7.1. Загрузка результатов экспериментального pipeline (Loading Experiment Pipeline Results)**

In [ ]:
metrics = pd.read_csv(EXPERIMENT_METRICS_PATH)
comparison = pd.read_csv(EXPERIMENT_COMPARISON_PATH)
predictions = pd.read_csv(VALIDATION_PREDICTIONS_PATH, parse_dates=["date_time"])
selected_pipeline = joblib.load(SELECTED_PIPELINE_PATH)

selected_predictions = predictions.query(
    "model == @SELECTED_MODEL and split == 'validation'"
).copy()

display(Markdown("### **Схема загруженных выходных данных экспериментального pipeline**"))
display(
    pd.DataFrame(
        {
            "output": ["metrics", "comparison", "validation_predictions"],
            "rows": [len(metrics), len(comparison), len(predictions)],
            "columns": [metrics.shape[1], comparison.shape[1], predictions.shape[1]],
        }
    )
)

display(Markdown("### **Validation metrics selected-модели**"))
display(comparison.query("model == @SELECTED_MODEL"))

## **7.2. Временной ряд транспортной нагрузки (Traffic Volume Time Series)**

In [ ]:
display(Markdown("### **Traffic volume time series: validation period**"))
figure, _ = plot_traffic_volume_time_series(
    selected_predictions,
    model_name=SELECTED_MODEL,
    output_path=TRAFFIC_VOLUME_TIME_SERIES_FIGURE_PATH,
)
plt.show()

## **7.3. Фактические и прогнозные значения (Actual and Predicted Values)**

In [ ]:
display(Markdown("### **Actual vs predicted traffic volume**"))
figure, _ = plot_actual_vs_predicted(
    selected_predictions,
    model_name=SELECTED_MODEL,
    output_path=ACTUAL_VS_PREDICTED_FIGURE_PATH,
)
plt.show()

display(Markdown("### **Observed vs predicted values**"))
figure, _ = plot_true_vs_predicted(
    selected_predictions,
    model_name=SELECTED_MODEL,
    output_path=TRUE_VS_PREDICTED_FIGURE_PATH,
)
plt.show()

## **7.4. Анализ ошибок прогноза (Prediction Error Analysis)**

In [ ]:
prediction_errors = add_prediction_errors(selected_predictions)
hourly_error = summarize_error_by_hour(selected_predictions, model_name=SELECTED_MODEL)

display(Markdown("### **Residual distribution**"))
figure, _ = plot_residual_distribution(
    selected_predictions,
    model_name=SELECTED_MODEL,
    output_path=RESIDUAL_DISTRIBUTION_FIGURE_PATH,
)
plt.show()

display(Markdown("### **Mean absolute error by hour**"))
figure, _ = plot_error_by_hour(
    selected_predictions,
    model_name=SELECTED_MODEL,
    output_path=ERROR_BY_HOUR_FIGURE_PATH,
)
plt.show()

display(hourly_error.sort_values("mae", ascending=False).head())

## **7.5. Сравнение моделей по validation-метрикам (Validation Model Comparison)**

In [ ]:
display(Markdown("### **Validation RMSE model comparison**"))
figure, _ = plot_model_comparison(
    comparison,
    metric="rmse",
    output_path=MODEL_COMPARISON_FIGURE_PATH,
)
plt.show()

display(comparison[["validation_rank", "model", "mae", "rmse", "mape", "r2"]])

## **7.6. Важность преобразованных признаков (Transformed Feature Importance)**

In [ ]:
feature_importance = extract_feature_importance(selected_pipeline)

display(Markdown("### **Top transformed feature importances: CatBoostRegressor**"))
figure, _ = plot_feature_importance(
    selected_pipeline,
    top_n=15,
    output_path=FEATURE_IMPORTANCE_FIGURE_PATH,
)
plt.show()

display(feature_importance.head(15))

## **7.7. Аудит методологических ограничений (Methodological Constraints Audit)**

In [ ]:
figure_paths = (
    TRAFFIC_VOLUME_TIME_SERIES_FIGURE_PATH,
    ACTUAL_VS_PREDICTED_FIGURE_PATH,
    RESIDUAL_DISTRIBUTION_FIGURE_PATH,
    TRUE_VS_PREDICTED_FIGURE_PATH,
    ERROR_BY_HOUR_FIGURE_PATH,
    MODEL_COMPARISON_FIGURE_PATH,
    FEATURE_IMPORTANCE_FIGURE_PATH,
)
methodological_audit = pd.DataFrame(
    {
        "check": [
            "Predictions contain validation split only",
            "Comparison contains validation metrics only",
            "Selected persisted pipeline is CatBoostRegressor",
            "No model training is performed in this notebook",
            "All visualization figures were generated",
        ],
        "passed": [
            set(predictions["split"]) == {"validation"},
            set(comparison["split"]) == {"validation"},
            type(selected_pipeline.named_steps["model"]).__name__ == "CatBoostRegressor",
            True,
            all(path.is_file() for path in figure_paths),
        ],
    }
)

display(Markdown("### **Methodological audit results**"))
display(methodological_audit)

assert bool(methodological_audit["passed"].all()), (
    "Model result visualization methodology audit failed."
)

## **7.8. Анализ и интерпретация результатов визуализации (Analysis and Interpretation of Visualization Results)**

На этапе **Visualization and Model Result Analysis** была выполнена визуальная оценка результатов прогнозирования транспортной нагрузки на основе выходных данных проведенного экспериментального исследования. В качестве исходных результатов использовались сохраненные validation-прогнозы, таблица сравнения моделей по метрикам качества и сохраненный pipeline модели `CatBoostRegressor`, показавшей лучший результат по validation RMSE.

**Ключевые результаты:**

1. **Динамика транспортной нагрузки имеет выраженную временную структуру.**
   График временного ряда фактических значений `traffic_volume` показывает наличие повторяющихся суточных и недельных колебаний транспортной нагрузки. Это подтверждает корректность использования временных, циклических, лаговых и скользящих признаков при построении модели прогнозирования. Нагрузка изменяется не случайным образом, а зависит от регулярных временных паттернов, что соответствует предметной логике задачи прогнозирования транспортного потока.

2. **Модель `CatBoostRegressor` хорошо воспроизводит общую динамику транспортного потока.**
   График сопоставления фактических и прогнозных значений показывает, что прогнозная кривая в целом повторяет форму реального временного ряда. Модель корректно улавливает основные пики и спады транспортной нагрузки, что подтверждает ее способность учитывать нелинейные зависимости между признаками и целевой переменной. При этом отдельные локальные отклонения сохраняются, что объясняется высокой изменчивостью транспортного потока в отдельные часы.

3. **Диаграмма `y_true` vs `y_pred` подтверждает высокую согласованность прогнозов с фактическими значениями.**
   На графике сопоставления наблюдаемых и предсказанных значений точки располагаются преимущественно вдоль диагонали идеального прогноза. Это означает, что модель не только повторяет временную динамику, но и в большинстве случаев формирует численные прогнозы, близкие к фактическим значениям транспортной нагрузки. Наличие небольшого рассеивания вокруг диагонали отражает естественные ошибки модели, связанные с резкими изменениями транспортного потока.

4. **Распределение остатков показывает отсутствие выраженного систематического смещения.**
   Распределение residuals сосредоточено около нулевого значения, что указывает на отсутствие сильного систематического завышения или занижения прогноза. При этом наличие хвостов распределения показывает, что в отдельных наблюдениях модель допускает более крупные ошибки. Такие ошибки могут быть связаны с резкими изменениями дорожной ситуации, нестандартными погодными условиями, праздничными периодами или иными факторами, которые не полностью отражены в исходных признаках.

5. **Ошибка прогноза зависит от времени суток.**
   График средней абсолютной ошибки по часам показывает, что качество прогноза неодинаково в течение суток. В отдельные часы, особенно в периоды интенсивного изменения транспортного потока, ошибка возрастает. Это является важным результатом для транспортной предметной области, поскольку пиковые периоды и переходные интервалы между низкой и высокой нагрузкой сложнее прогнозировать, чем стабильные ночные или дневные участки.

6. **Сравнение моделей подтверждает преимущество ансамблевых методов.**
   График сравнения моделей по validation RMSE показывает, что ансамблевые модели демонстрируют более высокое качество прогнозирования по сравнению с простыми базовыми моделями. Наилучший результат среди рассмотренных моделей показал `CatBoostRegressor`, что подтверждает целесообразность использования ансамблевых методов машинного обучения для задачи прогнозирования нагрузки автотранспортной системы. Это согласуется с рабочей гипотезой исследования о способности ансамблей эффективнее учитывать сложные нелинейные зависимости в данных.

7. **Анализ важности признаков подтверждает значимость исторических и временных факторов.**
   График feature importance для сохраненного `CatBoostRegressor` pipeline показывает, что наибольший вклад в прогноз вносят лаговые признаки транспортной нагрузки, прежде всего значения нагрузки в предыдущие часы и аналогичные периоды прошлых суток или недели. Также значимыми являются циклические признаки времени, отражающие суточную периодичность. Это подтверждает, что модель опирается не на случайные зависимости, а на содержательно интерпретируемые факторы, связанные с временной природой транспортного потока.

**Итоговое методологическое резюме:** этап **Visualization and Model Result Analysis** позволил визуально подтвердить качество работы разработанного pipeline прогнозирования транспортной нагрузки. Полученные графики показывают, что лучшая модель `CatBoostRegressor` корректно воспроизводит основную динамику транспортного потока, имеет относительно небольшие ошибки на validation-выборке и опирается на содержательно значимые временные и исторические признаки. Результаты визуализации подтверждают применимость ансамблевых методов машинного обучения для прогнозирования нагрузки автотранспортной системы и формируют основу для дальнейшего анализа качества, ограничений и практической применимости модели на последующих этапах исследования.
